# 1. Import & Load Data

In [1]:
# Cell 1 - Import & Load Data
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('../data/saca_final_dataset_self.csv')

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nSeverity distribution:")
print(df['Severity'].value_counts(normalize=True))

Dataset loaded successfully!
Shape: (29995, 256)

Severity distribution:
Severity
Moderate    0.333389
Mild        0.333322
Severe      0.333289
Name: proportion, dtype: float64


# 2. Exploratory Data Analysis

In [2]:
# Cell 2 â€” Exploratory Data Analysis
print("=== Basic EDA for SACA Dataset ===")
print(f"Total samples (rows): {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")
print(f"Number of unique diseases: {df['diseases'].nunique()}")

# Symptom columns only (exclude targets)
symptom_cols = [col for col in df.columns if col not in ['diseases', 'Severity']]

# Sparsity
sparsity = (df[symptom_cols] == 0).mean().mean() * 100
print(f"Sparsity (percentage of zeros in symptoms): {sparsity:.2f}%")

# Most common symptoms
symptom_counts = df[symptom_cols].sum().sort_values(ascending=False)
print("\nTop 10 most common symptoms:")
print(symptom_counts.head(10))

print("\n10 rarest symptoms:")
print(symptom_counts.tail(10))

=== Basic EDA for SACA Dataset ===
Total samples (rows): 29,995
Total columns: 256
Number of unique diseases: 211
Sparsity (percentage of zeros in symptoms): 97.81%

Top 10 most common symptoms:
vomiting                4604
sharp abdominal pain    4394
nausea                  4182
sharp chest pain        3824
cough                   3788
shortness of breath     3404
fever                   3139
weakness                3070
headache                2973
back pain               2884
dtype: int64

10 rarest symptoms:
impotence                           40
bumps on penis                      38
infertility                         38
early or late onset of menopause    38
irregular belly button              36
flushing                            30
symptoms of prostate                24
bladder mass                        19
neck mass                           16
itchy ear(s)                        12
dtype: int64


# 3.  Preprocessing

In [3]:
# Cell 3 - Preprocessing & Robust Split
from sklearn.model_selection import train_test_split
import pandas as pd

# Prepare features and targets
X = df.drop(['diseases', 'Severity'], axis=1)  # 254 symptom features
y_disease = df['diseases']                      # 211-class target
y_severity = df['Severity']                     # 3-class (Mild/Moderate/Severe)

print("X and y prepared successfully!")
print(f"Feature matrix shape: {X.shape}")
print(f"Disease target classes: {y_disease.nunique()}")
print(f"Severity target classes: {y_severity.nunique()}")

# Handle rare classes (threshold = 10)
class_counts = y_disease.value_counts()
rare_classes = class_counts[class_counts < 10].index

rare_mask = y_disease.isin(rare_classes)
X_rare    = X[rare_mask]
y_rare_d  = y_disease[rare_mask]
X_common  = X[~rare_mask]
y_common_d = y_disease[~rare_mask]

# Stratified split on common classes only
X_train_c, X_temp, y_train_d_c, y_temp_d = train_test_split(
    X_common, y_common_d, test_size=0.3, stratify=y_common_d, random_state=42
)
X_val, X_test, y_val_d, y_test_d = train_test_split(
    X_temp, y_temp_d, test_size=0.5, stratify=y_temp_d, random_state=42
)

# Add rare classes into training set
X_train  = pd.concat([X_train_c, X_rare], axis=0, ignore_index=True)
y_train_d = pd.concat([y_train_d_c, y_rare_d], axis=0, ignore_index=True)

# Align severity labels
y_train_s = y_severity.loc[X_train.index].reset_index(drop=True)
y_val_s   = y_severity.loc[X_val.index].reset_index(drop=True)
y_test_s  = y_severity.loc[X_test.index].reset_index(drop=True)

print("\nRobust split completed!")
print(f"Train set:      {X_train.shape[0]:,} samples")
print(f"Validation set: {X_val.shape[0]:,} samples")
print(f"Test set:       {X_test.shape[0]:,} samples")
print(f"Rare classes moved to train: {len(rare_classes)}")

X and y prepared successfully!
Feature matrix shape: (29995, 254)
Disease target classes: 211
Severity target classes: 3

Robust split completed!
Train set:      21,042 samples
Validation set: 4,476 samples
Test set:       4,477 samples
Rare classes moved to train: 31


# 4. SMOTE with single-sample classes

In [4]:
# Cell 4 - SMOTE Balancing
from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd

print("Applying SMOTE on training set only...")

# Step 1: Remove single-sample classes (SMOTE needs at least 2)
class_counts = Counter(y_train_d)
valid_classes  = [cls for cls, count in class_counts.items() if count >= 2]
removed_classes = [cls for cls, count in class_counts.items() if count < 2]

print(f"Removing {len(removed_classes)} single-sample classes...")

# Filter training data
mask = y_train_d.isin(valid_classes)
X_train_filtered = X_train[mask].reset_index(drop=True)
y_train_filtered = y_train_d[mask].reset_index(drop=True)

# Step 2: Auto-adjust k_neighbors safely
min_samples = min(Counter(y_train_filtered).values())
k = max(1, min(5, min_samples - 1))
print(f"Minimum class size: {min_samples} â†’ using k_neighbors={k}")

# Step 3: Apply SMOTE
smote = SMOTE(random_state=42, k_neighbors=k)
resampled_data = smote.fit_resample(
    X_train_filtered, y_train_filtered
)
X_train_resampled = resampled_data[0]
y_train_d_resampled = resampled_data[1]

print(f"\nBefore SMOTE: {X_train_filtered.shape[0]:,} samples")
print(f"After SMOTE:  {X_train_resampled.shape[0]:,} samples")
print(f"Synthetic samples added: {X_train_resampled.shape[0] - X_train_filtered.shape[0]:,}")
print(f"Removed single-sample classes: {removed_classes}")
print("\nClass distribution after SMOTE (top 10):")
print(y_train_d_resampled.value_counts().head(10))

Applying SMOTE on training set only...
Removing 4 single-sample classes...
Minimum class size: 2 â†’ using k_neighbors=1

Before SMOTE: 21,038 samples
After SMOTE:  91,908 samples
Synthetic samples added: 70,870
Removed single-sample classes: ['breast cancer', 'dengue fever', 'necrotizing fasciitis', 'leukemia']

Class distribution after SMOTE (top 10):
diseases
heart failure       444
delirium            444
marijuana abuse     444
flu                 444
sprain or strain    444
osteoarthritis      444
impetigo            444
gum disease         444
inguinal hernia     444
psoriasis           444
Name: count, dtype: int64


# 5. Group Split by Symptom Pattern

#### A. example (Prevent Leakage IF we this concern)

In [5]:
# Cell 5 - Group Split by Symptom Pattern (Prevent Leakage)
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Use resampled data from Cell 4
X_train_resampled_df = pd.DataFrame(
    X_train_resampled, 
    columns=X_train.columns
).copy()  # fixes PerformanceWarning

# Create symptom pattern group
symptom_cols = [col for col in X_train_resampled_df.columns 
                if col not in ['diseases', 'Severity']]

X_train_resampled_df['symptom_pattern'] = (
    X_train_resampled_df[symptom_cols]
    .astype(str)
    .agg(''.join, axis=1)
)

# Group split to prevent data leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
split_indices = next(gss.split(
    X_train_resampled_df,
    y_train_d_resampled,
    groups=X_train_resampled_df['symptom_pattern']
))
train_idx = split_indices[0]
temp_idx = split_indices[1]

# Final clean training set
X_train_final = (
    X_train_resampled_df
    .iloc[train_idx]
    .drop(columns=['symptom_pattern'])
    .reset_index(drop=True)
)
y_train_d_final = y_train_d_resampled.iloc[train_idx].reset_index(drop=True)

# Keep validation and test as they are
X_val_final  = X_val.copy()
X_test_final = X_test.copy()

print("Group split completed!")
print(f"Final train size: {X_train_final.shape[0]:,} samples")
print(f"Validation size:  {X_val_final.shape[0]:,} samples")
print(f"Test size:        {X_test_final.shape[0]:,} samples")

Group split completed!
Final train size: 65,050 samples
Validation size:  4,476 samples
Test size:        4,477 samples


# 6. Models Training Setup

In [6]:
# Cell 6 - Setup (Label Encoding + Clean Column Names)
from sklearn.preprocessing import LabelEncoder
import re

# Clean column names (remove special characters)
def clean_col_names(df):
    df = df.copy()
    df.columns = [re.sub(r'[^A-Za-z0-9_]', '_', str(col)) for col in df.columns]
    return df

# Apply cleaning
X_train = clean_col_names(X_train_final)
X_val   = clean_col_names(X_val_final)
X_test  = clean_col_names(X_test_final)

# Assign targets
y_train = y_train_d_final
y_val   = y_val_d
y_test  = y_test_d

# Encode labels (convert disease names to numbers)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc   = le.transform(y_val)
y_test_enc  = le.transform(y_test)

num_classes = len(le.classes_)

print("Setup completed!")
print(f"Total classes:   {num_classes}")
print(f"Train size:      {X_train.shape[0]:,} samples")
print(f"Validation size: {X_val.shape[0]:,} samples")
print(f"Test size:       {X_test.shape[0]:,} samples")
print(f"\nSample disease labels: {list(le.classes_[:5])}")

Setup completed!
Total classes:   207
Train size:      65,050 samples
Validation size: 4,476 samples
Test size:       4,477 samples

Sample disease labels: ['abdominal aortic aneurysm', 'abdominal hernia', 'acne', 'acute bronchitis', 'acute kidney injury']


# 7. Decision Tree (Baseline Model)

In [7]:
# Cell 7 - Decision Tree (Baseline Model)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import time

print(" Training Decision Tree baseline...")
start_dt = time.time()

# Train
dt = DecisionTreeClassifier(
    random_state=42,
    max_depth=10
)
dt.fit(X_train, y_train_enc)

dt_time = time.time() - start_dt

# Evaluate
dt_val_preds  = dt.predict(X_val)
dt_test_preds = dt.predict(X_test)

dt_val_acc  = accuracy_score(y_val_enc,  dt_val_preds)
dt_test_acc = accuracy_score(y_test_enc, dt_test_preds)

print(f" Decision Tree trained in {dt_time:.2f}s")
print(f"Validation Accuracy: {dt_val_acc:.4f} ({dt_val_acc*100:.2f}%)")
print(f"Test Accuracy:       {dt_test_acc:.4f} ({dt_test_acc*100:.2f}%)")

 Training Decision Tree baseline...
 Decision Tree trained in 0.78s
Validation Accuracy: 0.0279 (2.79%)
Test Accuracy:       0.0353 (3.53%)


# 8. LightGBM Training

In [8]:
# Cell 8 - LightGBM Training
import lightgbm as lgb
from sklearn.metrics import accuracy_score
import time

print(" Training LightGBM...")
start_lgb = time.time()

# Parameters
lgb_params = {
    'objective':        'multiclass',
    'num_class':        num_classes,
    'metric':           'multi_logloss',
    'learning_rate':    0.05,
    'num_leaves':       31,
    'min_data_in_leaf': 20,
    'feature_fraction': 0.75,
    'device':           'cpu',
    'verbose':          -1,
    'random_state':     42
}

# Datasets
lgb_train = lgb.Dataset(X_train, label=y_train_enc)
lgb_val   = lgb.Dataset(X_val,   label=y_val_enc, reference=lgb_train)

# Train
model_lgb = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=500,
    valid_sets=[lgb_val],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)

lgb_time = time.time() - start_lgb

# Evaluate
import numpy as np
lgb_val_prob = np.asarray(model_lgb.predict(X_val))
lgb_test_prob = np.asarray(model_lgb.predict(X_test))
lgb_val_preds = np.argmax(lgb_val_prob, axis=1)
lgb_test_preds = np.argmax(lgb_test_prob, axis=1)

lgb_val_acc  = accuracy_score(y_val_enc,  lgb_val_preds)
lgb_test_acc = accuracy_score(y_test_enc, lgb_test_preds)

print(f"\n LightGBM trained in {lgb_time:.2f}s")
print(f"Best iteration:      {model_lgb.best_iteration}")
print(f"Validation Accuracy: {lgb_val_acc:.4f} ({lgb_val_acc*100:.2f}%)")
print(f"Test Accuracy:       {lgb_test_acc:.4f} ({lgb_test_acc*100:.2f}%)")

 Training LightGBM...
Training until validation scores don't improve for 50 rounds
[50]	valid_0's multi_logloss: 0.729652
[100]	valid_0's multi_logloss: 0.673926
[150]	valid_0's multi_logloss: 0.704478
Early stopping, best iteration is:
[103]	valid_0's multi_logloss: 0.673368

 LightGBM trained in 66.29s
Best iteration:      103
Validation Accuracy: 0.8472 (84.72%)
Test Accuracy:       0.8465 (84.65%)


# 9. CatBoost:

In [9]:
# Cell 9 - CatBoost Training
import catboost as cb
from sklearn.metrics import accuracy_score
import time

print(" Training CatBoost...")
print("  This will take ~11 minutes, please wait...")
start_cb = time.time()

# Train
model_cb = cb.CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='MultiClass',
    verbose=100,
    random_seed=42,
    task_type='CPU'
)

model_cb.fit(
    X_train, y_train_enc,
    eval_set=(X_val, y_val_enc),
    early_stopping_rounds=50
)

cb_time = time.time() - start_cb

# Evaluate
cb_val_preds  = model_cb.predict(X_val).flatten()
cb_test_preds = model_cb.predict(X_test).flatten()

cb_val_acc  = accuracy_score(y_val_enc,  cb_val_preds)
cb_test_acc = accuracy_score(y_test_enc, cb_test_preds)

print(f"\n CatBoost trained in {cb_time:.2f}s")
print(f"Validation Accuracy: {cb_val_acc:.4f} ({cb_val_acc*100:.2f}%)")
print(f"Test Accuracy:       {cb_test_acc:.4f} ({cb_test_acc*100:.2f}%)")

 Training CatBoost...
  This will take ~11 minutes, please wait...
0:	learn: 5.1904324	test: 5.2542349	best: 5.2542349 (0)	total: 2.1s	remaining: 17m 28s
100:	learn: 1.9540100	test: 1.9156103	best: 1.9156103 (100)	total: 3m 16s	remaining: 12m 57s
200:	learn: 1.2635381	test: 1.2052841	best: 1.2052841 (200)	total: 6m 24s	remaining: 9m 32s
300:	learn: 0.9104580	test: 0.8809407	best: 0.8809407 (300)	total: 9m 36s	remaining: 6m 21s
400:	learn: 0.7063751	test: 0.7035686	best: 0.7035686 (400)	total: 12m 48s	remaining: 3m 9s
499:	learn: 0.5713966	test: 0.5901699	best: 0.5901699 (499)	total: 15m 50s	remaining: 0us

bestTest = 0.5901699102
bestIteration = 499


 CatBoost trained in 951.67s
Validation Accuracy: 0.8950 (89.50%)
Test Accuracy:       0.8841 (88.41%)


# 10. Evaluation + Save Models + JSON Test

In [10]:
# Cell 10 - Evaluation + Save Models + JSON Test
import json
import numpy as np
import joblib
from sklearn.metrics import accuracy_score, classification_report

required_objects = [
    'dt', 'model_lgb', 'model_cb',
    'X_test', 'y_test_enc', 'le'
]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError(
        "Missing trained objects: " + ", ".join(missing_objects) +
        ". Run the notebook from the data-preparation cells through "
        "Cells 7, 8, and 9 before running the final evaluation cell."
    )

print("=" * 50)
print(" FINAL MODEL EVALUATION")
print("=" * 50)

# --- Decision Tree ---
dt_test_preds = dt.predict(X_test)
dt_test_acc   = accuracy_score(y_test_enc, dt_test_preds)

# --- LightGBM ---
lgb_test_prob = np.asarray(model_lgb.predict(X_test))
lgb_test_preds = np.argmax(lgb_test_prob, axis=1)
lgb_test_acc   = accuracy_score(y_test_enc, lgb_test_preds)

# --- CatBoost ---
cb_test_preds = model_cb.predict(X_test).flatten()
cb_test_acc   = accuracy_score(y_test_enc, cb_test_preds)

# --- Summary ---
print(f"\n{'Model':<15} {'Test Accuracy':>15}")
print("-" * 32)
print(f"{'Decision Tree':<15} {dt_test_acc*100:>14.2f}%")
print(f"{'LightGBM':<15} {lgb_test_acc*100:>14.2f}%")
print(f"{'CatBoost':<15} {cb_test_acc*100:>14.2f}%  BEST")

# --- Save Models ---
print("\n Saving models...")
joblib.dump(dt,       '../models/decision_tree.pkl')
joblib.dump(model_lgb,'../models/lightgbm_model.pkl')
joblib.dump(le,       '../models/label_encoder.pkl')
model_cb.save_model(  '../models/catboost_model.cbm')
print(" All models saved to /models folder!")

# --- Create JSON Test File ---
print("\n Creating JSON test file...")

# Take 10 real samples from test set
sample_indices = list(range(10))
test_cases = []

for i in sample_indices:
    symptoms = X_test.iloc[i].to_dict()
    true_label_enc = int(y_test_enc[i])
    true_disease = le.inverse_transform([true_label_enc])[0]

    # LightGBM prediction
    lgb_prob = np.asarray(model_lgb.predict(X_test.iloc[[i]]))
    lgb_pred = le.inverse_transform([np.argmax(lgb_prob)])[0]

    # CatBoost prediction
    cb_pred = le.inverse_transform(
        model_cb.predict(X_test.iloc[[i]]).flatten()
    )[0]

    test_cases.append({
        "sample_id": i,
        "symptoms": symptoms,
        "true_disease": true_disease,
        "predictions": {
            "lightgbm": lgb_pred,
            "catboost": cb_pred
        },
        "correct": {
            "lightgbm": lgb_pred == true_disease,
            "catboost": cb_pred == true_disease
        }
    })

# Save JSON
with open('../data/test_results.json', 'w') as f:
    json.dump(test_cases, f, indent=4)

print(" JSON test file saved to /data/test_results.json!")
print("\n Sample predictions:")
for case in test_cases:
    print(f"\nSample {case['sample_id']}:")
    print(f"  True disease : {case['true_disease']}")
    print(f"  LightGBM pred: {case['predictions']['lightgbm']} {'' if case['correct']['lightgbm'] else ''}")
    print(f"  CatBoost pred: {case['predictions']['catboost']} {'' if case['correct']['catboost'] else ''}")

print("\n" + "=" * 50)
print(" ALL DONE! ML pipeline is completed!")
print("=" * 50)

 FINAL MODEL EVALUATION

Model             Test Accuracy
--------------------------------
Decision Tree             3.53%
LightGBM                 84.65%
CatBoost                 88.41%  BEST

 Saving models...
 All models saved to /models folder!

 Creating JSON test file...
 JSON test file saved to /data/test_results.json!

 Sample predictions:

Sample 0:
  True disease : stroke
  LightGBM pred: stroke 
  CatBoost pred: stroke 

Sample 1:
  True disease : hemiplegia
  LightGBM pred: hemiplegia 
  CatBoost pred: hemiplegia 

Sample 2:
  True disease : appendicitis
  LightGBM pred: appendicitis 
  CatBoost pred: appendicitis 

Sample 3:
  True disease : gastrointestinal hemorrhage
  LightGBM pred: gastrointestinal hemorrhage 
  CatBoost pred: gastrointestinal hemorrhage 

Sample 4:
  True disease : coronary atherosclerosis
  LightGBM pred: cardiomyopathy 
  CatBoost pred: cardiomyopathy 

Sample 5:
  True disease : impetigo
  LightGBM pred: impetigo 
  CatBoost pred: impetigo 

Sample 